# Live Headline Index Polling

## Integration Guide: Pre-Aggregated Sentiment Index

This notebook demonstrates how to integrate the **Permutable AI Live Headline Index API** into an analytical workflow. The index endpoint returns sentiment that has already been aggregated into hourly buckets, making it easier to consume directly in research and monitoring workflows without client-side aggregation.

> **Disclaimer:** This notebook is provided for informational and research purposes only. Nothing in this notebook constitutes financial advice or a recommendation to buy, sell, or hold any asset. Sentiment indicators derived here reflect aggregated model outputs and should not be used as the sole basis for any investment decision.

### How the Index Differs from the Raw Feed

| | Raw Feed (`/headlines/feed/live`) | Index (`/headlines/index/ticker/live`) |
|---|---|---|
| **Granularity** | One row per headline | One row per (ticker, hour, topic) |
| **Key metrics** | Per-headline `sentiment_score`, probabilities | `sentiment_sum`, `sentiment_abs_sum`, `headline_count`, `sentiment_std` |
| **Client work** | Must aggregate before use in your workflow | Ready to use directly |
| **Best for** | Research, custom aggregations, NLP feature engineering | Monitoring dashboards, sentiment tracking, downstream integration |

### What the Endpoint Returns

`GET /v1/headlines/index/ticker/live/{ticker}` returns the latest 2 hours of pre-aggregated sentiment index data. Each record maps to the `HeadlineIndex` schema:

| Field | Type | Description |
|---|---|---|
| `ticker` | `str` | Asset ticker (e.g. `BZ_COM`) |
| `publication_time` | `datetime` | Start or end of the hourly bucket |
| `topic_name` | `str` | Sentiment topic |
| `index_type` | `str` | `EXPLICIT` / `IMPLICIT` / `COMBINED` |
| `headline_count` | `int` | Number of headlines in this bucket |
| `sentiment_sum` | `float` | Sum of sentiment scores (net sentiment score) |
| `sentiment_abs_sum` | `float` | Sum of absolute sentiment scores (total sentiment magnitude) |
| `sentiment_std` | `float` | Std dev of sentiment scores (dispersion / disagreement) |

**Derived metric — Average Sentiment:**

```
sentiment_avg = sentiment_sum / headline_count   ∈ [−1, +1]
```

An average sentiment near +1 means all headlines in the bucket were strongly positive; near −1 means all were strongly negative. Values near 0 indicate mixed or neutral sentiment regardless of volume.

### Polling Architecture

```
┌─────────────────────────────────────────────────────────────────────────┐
│  Every 15 minutes                                                       │
│                                                                         │
│  [API] ──► fetch_live_index(ticker)                                     │
│                  │                                                      │
│                  ▼                                                      │
│            upsert_index()  ──► [SQLite: headline_index]                 │
│                                       │                                 │
│                                       ▼                                 │
│                        load from DB ──► normalised sentiment indicator  │
└─────────────────────────────────────────────────────────────────────────┘
```

**Upsert key:** `(ticker, publication_time, topic_name, index_type)`. The live endpoint always returns the current 2-hour window, so successive polls overlap. Upserting on this composite key keeps the database deduplicated without bookkeeping.

### Pipeline Steps

1. **Install** dependencies and configure API credentials
2. **Define** the fetch and upsert functions
3. *(Optional)* **Historical backfill** — populate the database with up to 90 days of history before starting live polling. Recommended if you are running a monitoring dashboard and want meaningful trend charts from the first run.
4. **Dry-run** a single poll to validate connectivity and inspect the index schema
5. **Visualise** the initial hourly snapshot (sentiment bars, news flow)
6. **Run** the 15-minute polling loop
7. **Visualise** the accumulated live window (multi-ticker index, heatmap, average sentiment band)
8. **Derive** a normalised average sentiment indicator for research and monitoring

### Prerequisites

- A Permutable AI API key (available from your account dashboard)
- Python 3.9+ with the packages listed in the cell below

In [ ]:
# Install required packages — run this cell once, then restart the kernel if needed
%pip install requests pandas matplotlib seaborn

In [ ]:
import getpass
import sqlite3
import time
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)
sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams.update({"figure.dpi": 120, "figure.figsize": (12, 5)})

In [ ]:
# ── API credentials ────────────────────────────────────────────────────────────
# The key is read via getpass so it is never displayed or stored in the notebook
API_KEY = getpass.getpass("Enter your Permutable AI API key: ")

# ── Endpoint configuration ─────────────────────────────────────────────────────
BASE_URL = "https://copilot-api.permutable.ai/v1"

# Edit this list to match the tickers included in your licence
TICKERS = ["BTC_CRY", "ETH_CRY", "BZ_COM", "EUR_IND"]

# Index type: COMBINED includes all headlines; EXPLICIT = direct mention; IMPLICIT = inferred
INDEX_TYPE = "COMBINED"  # EXPLICIT | IMPLICIT | COMBINED
TOPIC_PRESET = "ALL"  # topic preset name or "ALL"

# sparse=True returns only buckets that have headlines (recommended — avoids very large responses)
# align_to_period_end=True timestamps buckets at the close of each hour (e.g. 10:00 for 09:00–10:00)
SPARSE = True
ALIGN_TO_PERIOD_END = True

# ── Polling interval ───────────────────────────────────────────────────────────
POLL_INTERVAL_SECONDS = 15 * 60  # poll every 15 minutes

# ── Local storage ──────────────────────────────────────────────────────────────
DB_PATH = Path("headline_index.db")

print(f"Configured for {len(TICKERS)} tickers: {TICKERS}")
print(f"Index type       : {INDEX_TYPE}")
print(f"Polling interval : {POLL_INTERVAL_SECONDS // 60} minutes")
print(f"Local database   : {DB_PATH.resolve()}")

In [ ]:
def fetch_live_index(ticker: str) -> list[dict]:
    """
    Fetch the live headline sentiment index for a single ticker.

    Calls GET /v1/headlines/index/ticker/live/{ticker} and returns a flat list
    of record dicts matching the HeadlineIndex schema. A 'fetched_at' timestamp
    is appended so you can track when each poll retrieved each record.
    """
    url = f"{BASE_URL}/headlines/index/ticker/live/{ticker}"
    params = {
        "index_type": INDEX_TYPE,
        "topic_preset": TOPIC_PRESET,
        "sparse": str(SPARSE).lower(),
        "align_to_period_end": str(ALIGN_TO_PERIOD_END).lower(),
    }
    headers = {"x-api-key": API_KEY}

    response = requests.get(url, params=params, headers=headers, timeout=30)
    response.raise_for_status()

    records = response.json()
    for r in records:
        r["fetched_at"] = datetime.now(timezone.utc).isoformat()
    return records


# Quick connectivity check
print(f"Testing connectivity for {TICKERS[0]}...")
test_records = fetch_live_index(TICKERS[0])
print(f"  OK — {len(test_records)} index records returned")

In [ ]:
def get_connection() -> sqlite3.Connection:
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def setup_database() -> None:
    """Create the headline_index table and index if they do not already exist."""
    with get_connection() as conn:
        conn.execute("""
            CREATE TABLE IF NOT EXISTS headline_index (
                ticker              TEXT NOT NULL,
                publication_time    TEXT NOT NULL,
                topic_name          TEXT NOT NULL,
                index_type          TEXT NOT NULL,
                headline_count      INTEGER,
                sentiment_sum       REAL,
                sentiment_abs_sum   REAL,
                sentiment_std       REAL,
                fetched_at          TEXT,
                PRIMARY KEY (ticker, publication_time, topic_name, index_type)
            )
        """)
        conn.execute("""
            CREATE INDEX IF NOT EXISTS idx_hi_ticker_time
            ON headline_index (ticker, publication_time)
        """)
    print(f"Database ready: {DB_PATH.resolve()}")


setup_database()

In [ ]:
def upsert_index(records: list[dict]) -> int:
    """
    Upsert headline index records into the local SQLite database.

    INSERT OR REPLACE ensures that re-polling the same 2-hour window does not
    create duplicate rows. Records are uniquely identified by the composite key
    (ticker, publication_time, topic_name, index_type).

    Returns the number of rows written.
    """
    if not records:
        return 0

    columns = [
        "ticker",
        "publication_time",
        "topic_name",
        "index_type",
        "headline_count",
        "sentiment_sum",
        "sentiment_abs_sum",
        "sentiment_std",
        "fetched_at",
    ]
    placeholders = ", ".join(["?"] * len(columns))
    sql = f"INSERT OR REPLACE INTO headline_index ({', '.join(columns)}) " f"VALUES ({placeholders})"
    rows = [tuple(r.get(c) for c in columns) for r in records]

    with get_connection() as conn:
        conn.executemany(sql, rows)
    return len(rows)


print("upsert_index() defined")

## Historical Backfill *(Optional — Recommended for Dashboards)*

> **Skip this section if you only want live data.** Run it before the dry-run if you want your local database to contain multi-day history from the start — this gives monitoring dashboards and trend charts meaningful context immediately, rather than accumulating data hour-by-hour.

The live endpoint returns only the **most recent 2-hour window**. The historical endpoint (`/v1/headlines/index/ticker/historical/{ticker}`) lets you fetch up to **90 days** of past index data using keyset pagination, writing into the same SQLite table so records merge seamlessly with live-polled data.

| Property | Live | Historical |
|---|---|---|
| Window | Last 2 hours | Up to 90 days back |
| Pagination | None | Keyset via `next_token` |

Adjust `BACKFILL_DAYS` below and run the cell. The companion production app ([`app/live_index_polling`](../../../app/live_index_polling)) runs this backfill automatically on startup.

In [ ]:
from datetime import timedelta

BACKFILL_DAYS = 7  # days of history to fetch; max 90


def fetch_historical_index(ticker: str, start_date, end_date=None) -> list[dict]:
    """
    Paginate through GET /v1/headlines/index/ticker/historical/{ticker} and return
    all matching records as a flat list.

    Keyset pagination: each response includes 'has_more' and 'next_token'.
    We loop until has_more is False, passing next_token into every subsequent request.
    """
    url = f"{BASE_URL}/headlines/index/ticker/historical/{ticker}"
    params = {
        "start_date": start_date.isoformat(),
        "index_type": INDEX_TYPE,
        "topic_preset": TOPIC_PRESET,
        "sparse": str(SPARSE).lower(),
        "align_to_period_end": str(ALIGN_TO_PERIOD_END).lower(),
        "limit": 1000,
    }
    if end_date:
        params["end_date"] = end_date.isoformat()
    headers = {"x-api-key": API_KEY}

    all_records, page = [], 1
    while True:
        response = requests.get(url, params=params, headers=headers, timeout=30)
        response.raise_for_status()
        body = response.json()
        records = body["data"]
        for r in records:
            r["fetched_at"] = datetime.now(timezone.utc).isoformat()
        all_records.extend(records)
        print(f"    page {page:3d}: {len(records):5d} records  |  total so far: {len(all_records):6d}")
        if not body["has_more"]:
            break
        params["next_token"] = body["next_token"]
        page += 1
    return all_records


def backfill_all_tickers(days_back: int = BACKFILL_DAYS) -> None:
    """Fetch and upsert historical headline index for all configured tickers."""
    start = (datetime.utcnow() - timedelta(days=days_back)).date()
    print(f"Backfilling {len(TICKERS)} tickers from {start} ({days_back} days)\n")
    for ticker in TICKERS:
        print(f"  {ticker}")
        try:
            records = fetch_historical_index(ticker, start)
            n = upsert_index(records)
            print(f"    {n} rows upserted into headline_index\n")
        except requests.HTTPError as e:
            print(f"    HTTP {e.response.status_code} — skipping\n")
        except Exception as e:
            print(f"    {type(e).__name__}: {e}\n")


# ── Run backfill ───────────────────────────────────────────────────────────────
# Comment out the line below to skip the backfill and proceed directly to the dry run.
backfill_all_tickers()

## Dry Run

Fetch and store index data for all configured tickers once. Use this to verify your API key and inspect the raw schema before starting the polling loop.

In [ ]:
print("Running dry-run poll for all tickers...\n")
dry_run_rows = []

for ticker in TICKERS:
    try:
        records = fetch_live_index(ticker)
        n_written = upsert_index(records)
        dry_run_rows.append({"ticker": ticker, "records_fetched": len(records), "rows_written": n_written})
        print(f"  {ticker:12s}: {len(records):4d} index records | {n_written:4d} rows written")
    except requests.HTTPError as e:
        print(f"  {ticker}: HTTP {e.response.status_code} — {e.response.text[:200]}")
    except Exception as e:
        print(f"  {ticker}: {type(e).__name__}: {e}")

print()
display(pd.DataFrame(dry_run_rows))

# Preview stored data
with get_connection() as conn:
    sample = pd.read_sql(
        "SELECT * FROM headline_index ORDER BY publication_time DESC LIMIT 10",
        conn,
    )
print("\nSample rows from database:")
display(sample)

## Initial Visualisation

Inspect the first hourly snapshot: net sentiment by bucket and headline volume (news flow) per ticker.

In [ ]:
with get_connection() as conn:
    df = pd.read_sql("SELECT * FROM headline_index", conn)

df["publication_time"] = pd.to_datetime(df["publication_time"], utc=True)

if df.empty:
    print("No data yet — run the dry-run cell above first.")
else:
    active_tickers = [t for t in TICKERS if t in df["ticker"].unique()]
    colors = sns.color_palette("muted", len(active_tickers))

    # Aggregate to hourly buckets per ticker (sum across topics)
    hourly = (
        df.groupby(["ticker", "publication_time"])
        .agg(
            sentiment_sum=("sentiment_sum", "sum"),
            headline_count=("headline_count", "sum"),
        )
        .reset_index()
        .sort_values("publication_time")
    )
    hourly["sentiment_avg"] = hourly["sentiment_sum"] / hourly["headline_count"].replace(0, float("nan"))

    fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    # ── Plot 1: Net sentiment sum per ticker — hourly line ─────────────────
    ax = axes[0]
    for ticker, color in zip(active_tickers, colors):
        sub = hourly[hourly["ticker"] == ticker]
        ax.plot(sub["publication_time"], sub["sentiment_sum"], linewidth=1.6, label=ticker, color=color)
    ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
    ax.set_title("Net Sentiment Sum by Hour", fontsize=11, fontweight="bold")
    ax.set_ylabel("Sentiment Sum")
    ax.legend(fontsize=8, loc="upper right")

    # ── Plot 2: Headline count per ticker — hourly line ────────────────────
    ax = axes[1]
    for ticker, color in zip(active_tickers, colors):
        sub = hourly[hourly["ticker"] == ticker]
        ax.plot(sub["publication_time"], sub["headline_count"], linewidth=1.6, label=ticker, color=color)
    ax.set_title("Headline Count (News Flow) by Hour", fontsize=11, fontweight="bold")
    ax.set_ylabel("Headline Count")
    ax.set_xlabel("Date (UTC)")
    ax.legend(fontsize=8, loc="upper right")

    for ax in axes:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        ax.xaxis.set_major_locator(mdates.DayLocator())

    plt.suptitle(
        f"Initial Index Snapshot  —  {df['publication_time'].max().strftime('%Y-%m-%d %H:%M UTC')}",
        fontsize=12,
        fontweight="bold",
    )
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    # ── Plot 3: 5-hour rolling average sentiment — heatmap (ticker × hour) ──
    sentiment_avg_pivot = hourly.pivot_table(
        index="publication_time", columns="ticker", values="sentiment_avg", aggfunc="mean"
    ).sort_index()
    col_times = sentiment_avg_pivot.index  # preserve datetime index for tick logic
    rolling_sentiment_avg = sentiment_avg_pivot.rolling(window=5, min_periods=1).mean().T

    fig, ax = plt.subplots(figsize=(14, max(2.5, len(active_tickers) * 1.4)))
    sns.heatmap(
        rolling_sentiment_avg.rename(columns=lambda t: t.strftime("%b %d %H:%M")),
        ax=ax,
        cmap="RdYlGn",
        center=0,
        vmin=-1,
        vmax=1,
        linewidths=0.3,
        annot=False,
        cbar_kws={"label": "5-Hour Rolling Average Sentiment  [−1, +1]"},
    )
    # Show one tick per day at the first hour of each day
    seen_dates, day_positions, day_labels = set(), [], []
    for i, t in enumerate(col_times):
        if t.date() not in seen_dates:
            seen_dates.add(t.date())
            day_positions.append(i + 0.5)
            day_labels.append(t.strftime("%b %d"))
    ax.set_xticks(day_positions)
    ax.set_xticklabels(day_labels, rotation=30, ha="right", fontsize=8)
    ax.set_title("5-Hour Rolling Average Sentiment  —  Ticker × Hour", fontsize=12, fontweight="bold")
    ax.set_xlabel("Date (UTC)")
    ax.set_ylabel("Ticker")
    plt.tight_layout()
    plt.show()

    print(f"Total rows in database : {len(df)}")
    print(df.groupby("ticker")[["headline_count", "sentiment_sum"]].sum().round(4))

## 15-Minute Polling Loop

This cell runs indefinitely, polling all configured tickers every 15 minutes and upserting results into the local SQLite database.

**Stop:** press the ■ (Interrupt Kernel) button or use **Kernel → Interrupt**.

In production this logic would run as a scheduled job (cron, Airflow, or a Docker Compose service). This notebook form is suitable for research and supervised live monitoring.

In [ ]:
poll_count = 0
print(f"Starting polling loop — {len(TICKERS)} tickers every {POLL_INTERVAL_SECONDS // 60} min")
print("Interrupt the kernel to stop gracefully.\n")

try:
    while True:
        poll_count += 1
        t_start = datetime.now(timezone.utc)
        print(f"[Poll {poll_count}]  {t_start.strftime('%Y-%m-%d %H:%M:%S UTC')}")

        for ticker in TICKERS:
            try:
                records = fetch_live_index(ticker)
                n = upsert_index(records)
                print(f"  {ticker:12s}: {len(records):4d} records | {n:4d} upserted")
            except requests.HTTPError as e:
                print(f"  {ticker}: HTTP {e.response.status_code} — skipping, will retry next poll")
            except Exception as e:
                print(f"  {ticker}: {type(e).__name__}: {e}")

        elapsed = (datetime.now(timezone.utc) - t_start).total_seconds()
        wait = max(0.0, POLL_INTERVAL_SECONDS - elapsed)
        print(f"  Completed in {elapsed:.1f}s.  Next poll in {wait / 60:.1f} min.\n")
        time.sleep(wait)

except KeyboardInterrupt:
    print(f"\nPolling stopped after {poll_count} poll(s).")
    print(f"All data saved to: {DB_PATH.resolve()}")

## Post-Poll Visualisation

Run this cell after one or more polling cycles to explore the accumulated data:

1. `sentiment_sum` over time per ticker with ±1 std deviation band
2. Multi-ticker average sentiment comparison (`sentiment_sum / headline_count`)
3. Headline count heatmap — ticker × hour

In [ ]:
with get_connection() as conn:
    df = pd.read_sql("SELECT * FROM headline_index", conn)

df["publication_time"] = pd.to_datetime(df["publication_time"], utc=True)
df = df.sort_values("publication_time")

if df.empty:
    print("No data yet — run the polling loop above first.")
else:
    active_tickers = [t for t in TICKERS if t in df["ticker"].unique()]
    colors = sns.color_palette("muted", len(active_tickers))

    # Aggregate across topics to get one (ticker, hour) row for plotting
    ts = (
        df.groupby(["ticker", "publication_time"])
        .agg(
            sentiment_sum=("sentiment_sum", "sum"),
            sentiment_std=("sentiment_std", "mean"),
            headline_count=("headline_count", "sum"),
        )
        .reset_index()
        .sort_values("publication_time")
    )
    ts["sentiment_avg"] = ts["sentiment_sum"] / ts["headline_count"].replace(0, float("nan"))

    fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

    # ── Plot 1: Sentiment sum — hourly time series with std-dev band ───────
    ax = axes[0]
    for ticker, color in zip(active_tickers, colors):
        sub = ts[ts["ticker"] == ticker].sort_values("publication_time")
        ax.plot(sub["publication_time"], sub["sentiment_sum"], linewidth=1.6, label=ticker, color=color)
        ax.fill_between(
            sub["publication_time"],
            sub["sentiment_sum"] - sub["sentiment_std"].fillna(0),
            sub["sentiment_sum"] + sub["sentiment_std"].fillna(0),
            alpha=0.12,
            color=color,
        )
    ax.axhline(0, color="black", linewidth=0.7, linestyle="--")
    ax.set_title("Sentiment Sum by Hour", fontsize=12, fontweight="bold")
    ax.set_ylabel("Sentiment Sum")
    ax.legend(fontsize=8, loc="upper right")

    # ── Plot 2: Headline count — hourly time series ────────────────────────
    ax = axes[1]
    for ticker, color in zip(active_tickers, colors):
        sub = ts[ts["ticker"] == ticker].sort_values("publication_time")
        ax.plot(sub["publication_time"], sub["headline_count"], linewidth=1.6, label=ticker, color=color)
    ax.set_title("Headline Count (News Flow) by Hour", fontsize=12, fontweight="bold")
    ax.set_ylabel("Headline Count")
    ax.set_xlabel("Time (UTC)")
    ax.legend(fontsize=8, loc="upper right")

    for ax in axes:
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
        ax.xaxis.set_major_locator(mdates.DayLocator())

    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

    # ── Plot 3: 5-hour rolling average sentiment — heatmap (ticker × hour) ──
    sentiment_avg_pivot = ts.pivot_table(
        index="publication_time", columns="ticker", values="sentiment_avg", aggfunc="mean"
    ).sort_index()
    col_times = sentiment_avg_pivot.index  # preserve datetime index for tick logic
    rolling_sentiment_avg = sentiment_avg_pivot.rolling(window=5, min_periods=1).mean().T

    fig, ax = plt.subplots(figsize=(14, max(2.5, len(active_tickers) * 1.4)))
    sns.heatmap(
        rolling_sentiment_avg.rename(columns=lambda t: t.strftime("%b %d %H:%M")),
        ax=ax,
        cmap="RdYlGn",
        center=0,
        vmin=-1,
        vmax=1,
        linewidths=0.3,
        annot=False,
        cbar_kws={"label": "5-Hour Rolling Average Sentiment  [−1, +1]"},
    )
    # Show one tick per day at the first hour of each day
    seen_dates, day_positions, day_labels = set(), [], []
    for i, t in enumerate(col_times):
        if t.date() not in seen_dates:
            seen_dates.add(t.date())
            day_positions.append(i + 0.5)
            day_labels.append(t.strftime("%b %d"))
    ax.set_xticks(day_positions)
    ax.set_xticklabels(day_labels, rotation=30, ha="right", fontsize=8)
    ax.set_title("5-Hour Rolling Average Sentiment  —  Ticker × Hour", fontsize=12, fontweight="bold")
    ax.set_xlabel("Date (UTC)")
    ax.set_ylabel("Ticker")
    plt.tight_layout()
    plt.show()

## Sentiment Aggregation Example

Demonstrates how to load the accumulated index from the local database and derive a normalised sentiment indicator. This example is intended for **research and monitoring purposes only** and does not constitute financial advice.

**Aggregation logic:** aggregate topic-level index records to (ticker, hour) level, compute average sentiment, apply a rolling mean, then threshold to **HIGH / NEUTRAL / LOW**.

| Metric | Formula | Range |
|---|---|---|
| `sentiment_avg` | `sentiment_sum / headline_count` | [−1, +1] |
| `headline_count` | sum of headlines per bucket | ≥ 0 |
| `sentiment_std` | mean std dev across topics | ≥ 0 |

### Rolling Window

The smoothed average sentiment uses a **5-hour rolling mean** (`window=5`) — each hour's value is the average of the current hour and the 4 preceding hours. This dampens short-lived spikes caused by a single noisy headline bucket.

The window size is a matter of preference and depends on how quickly you want the indicator to react:

| `window` | Behaviour |
|---|---|
| 1 | No smoothing — raw hourly average sentiment |
| 3 | Light smoothing — responsive to intraday shifts |
| **5** | **Moderate smoothing (default)** — balances noise reduction and responsiveness |
| 12 | Heavy smoothing — captures half-day trends, slower to react |
| 24 | Daily smoothing — suitable for multi-day backfill analysis |

Adjust `ROLLING_WINDOW` in the code cell below to suit your use case.

In [ ]:
with get_connection() as conn:
    df = pd.read_sql("SELECT * FROM headline_index", conn)

df["publication_time"] = pd.to_datetime(df["publication_time"], utc=True)

# ── 1. Aggregate to (ticker, hour) ─────────────────────────────────────────────
df_agg = (
    df.groupby(["ticker", "publication_time"])
    .agg(
        sentiment_sum=("sentiment_sum", "sum"),
        sentiment_abs_sum=("sentiment_abs_sum", "sum"),
        headline_count=("headline_count", "sum"),
        sentiment_std=("sentiment_std", "mean"),
    )
    .reset_index()
    .sort_values(["ticker", "publication_time"])
)

# ── 2. Compute average sentiment and rolling mean ─────────────────────────────
ROLLING_WINDOW = 5  # hours — increase for smoother signal, decrease for faster reaction

df_agg["sentiment_avg"] = df_agg["sentiment_sum"] / df_agg["headline_count"].replace(0, float("nan"))
df_agg["sentiment_smooth"] = df_agg.groupby("ticker")["sentiment_avg"].transform(
    lambda x: x.rolling(ROLLING_WINDOW, min_periods=1).mean()
)

# ── 3. Threshold-based sentiment indicator ─────────────────────────────────────
UPPER_THRESHOLD = 0.40  # above this → HIGH sentiment
LOWER_THRESHOLD = -0.40  # below this → LOW sentiment

df_agg["indicator"] = "NEUTRAL"
df_agg.loc[df_agg["sentiment_smooth"] >= UPPER_THRESHOLD, "indicator"] = "HIGH"
df_agg.loc[df_agg["sentiment_smooth"] <= LOWER_THRESHOLD, "indicator"] = "LOW"

# ── 4. Latest indicator per ticker ─────────────────────────────────────────────
latest = (
    df_agg.sort_values("publication_time")
    .groupby("ticker")
    .last()
    .reset_index()[["ticker", "publication_time", "sentiment_avg", "headline_count", "sentiment_std", "indicator"]]
)
print("Latest sentiment indicators:")
display(latest)

# ── 5. Sentiment indicator heatmap ─────────────────────────────────────────────
if len(df_agg) > 1:
    INDICATOR_COLORS = {"HIGH": "#2ecc71", "NEUTRAL": "#bdc3c7", "LOW": "#e74c3c"}

    # Pivot hourly: rows = tickers, columns = publication_time (one per hour)
    col_times = sorted(df_agg["publication_time"].unique())
    indicator_pivot = df_agg.pivot_table(
        index="ticker", columns="publication_time", values="indicator", aggfunc="last"
    ).reindex(columns=col_times)

    # Build a colour array: green = HIGH, red = LOW, grey = NEUTRAL
    num_map = {"HIGH": 1, "NEUTRAL": 0, "LOW": -1}
    num_pivot = indicator_pivot.applymap(lambda v: num_map.get(v, 0) if pd.notna(v) else float("nan"))

    color_array = np.full((*num_pivot.shape, 4), fill_value=[0.74, 0.76, 0.78, 1.0])
    color_array[num_pivot.values == 1] = [0.18, 0.80, 0.44, 1.0]  # green
    color_array[num_pivot.values == -1] = [0.90, 0.30, 0.24, 1.0]  # red

    fig, ax = plt.subplots(figsize=(14, max(2.5, len(df_agg["ticker"].unique()) * 1.4)))
    ax.imshow(color_array, aspect="auto")

    # One tick per day at the first hour of each day
    seen_dates, day_positions, day_labels = set(), [], []
    for i, t in enumerate(col_times):
        d = pd.Timestamp(t).date()
        if d not in seen_dates:
            seen_dates.add(d)
            day_positions.append(i)
            day_labels.append(pd.Timestamp(t).strftime("%b %d"))
    ax.set_xticks(day_positions)
    ax.set_xticklabels(day_labels, rotation=30, ha="right", fontsize=8)

    ax.set_yticks(range(len(indicator_pivot.index)))
    ax.set_yticklabels(indicator_pivot.index, fontsize=9)
    ax.set_title(
        f"5-Hour Rolling Sentiment Indicator  (HIGH ≥ {UPPER_THRESHOLD} | LOW ≤ {LOWER_THRESHOLD})",
        fontsize=12,
        fontweight="bold",
    )
    ax.set_xlabel("Date (UTC)")
    ax.set_ylabel("Ticker")

    from matplotlib.patches import Patch

    legend_elements = [
        Patch(facecolor="#2ecc71", label=f"HIGH  (≥ {UPPER_THRESHOLD})"),
        Patch(facecolor="#bdc3c7", label="NEUTRAL"),
        Patch(facecolor="#e74c3c", label=f"LOW   (≤ {LOWER_THRESHOLD})"),
    ]
    ax.legend(handles=legend_elements, loc="upper right", fontsize=8, framealpha=0.9)
    plt.tight_layout()
    plt.show()

    # ── 6. Headline count — time series ────────────────────────────────────
    colors = sns.color_palette("muted", len(df_agg["ticker"].unique()))
    fig, ax = plt.subplots(figsize=(14, 4))
    for ticker, color in zip(df_agg["ticker"].unique(), colors):
        grp = df_agg[df_agg["ticker"] == ticker]
        ax.plot(grp["publication_time"], grp["headline_count"], linewidth=1.6, label=ticker, color=color)
    ax.set_title("Headline Count (News Flow) by Hour", fontsize=11, fontweight="bold")
    ax.set_xlabel("Date (UTC)")
    ax.set_ylabel("Headline Count")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.xaxis.set_major_locator(mdates.DayLocator())
    ax.legend(fontsize=8, loc="upper right")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()

## Next Steps: Production Deployment

This notebook is an **integration reference** — it covers the complete workflow from API authentication through to a normalised sentiment indicator in a single, runnable document.

For a production deployment, see the accompanying **Live Index Polling Application** in [`app/live_index_polling`](../../../app/live_index_polling), which packages this workflow into:

- A standalone **polling service** with configurable intervals and ticker lists, including automatic historical backfill on first start
- A **PostgreSQL** store replacing the local SQLite database for multi-process access and durability
- A lightweight **REST API** exposing the latest sentiment index data
- A **monitoring dashboard** for real-time observation of news flow, average sentiment, and sentiment drift over the full backfill window

### Choosing Between the Index and Raw Feed

- Use the **index endpoint** (this notebook) when you want a ready-to-use hourly aggregation with minimal client-side processing — ideal for monitoring dashboards and downstream integration.
- Use the **raw feed endpoint** (`notebooks/live/headline_sentiment_polling.ipynb`) when you need per-headline granularity, custom aggregation logic, or are building NLP features for a machine learning model.

### Related Resources

- [Permutable AI API Documentation](https://docs.permutable.ai)
- [`app/live_index_polling`](../../../app/live_index_polling) — containerised polling app for this notebook's workflow that you can base your production build
- `notebooks/backtesting/index_cross_ticker_signal_assessment.ipynb` — backtesting pipeline for historical analysis of the index sentiment data
- `notebooks/live/headline_sentiment_polling.ipynb` — equivalent guide using the per-headline raw feed endpoint